<a href="https://colab.research.google.com/github/latidore/Genesis/blob/main/quadric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import plotly.graph_objects as go
from skimage import measure

def get_equation_string(coeffs):
    """
    Generates a LaTeX string representation of the quadric surface equation.
    coeffs: [A, B, C, D, E, F, G, H, I, J]
    Equation: Ax^2 + By^2 + Cz^2 + Dxy + Eyz + Fzx + Gx + Hy + Iz + J = 0
    """
    terms = []
    # LaTeX representations for variables
    variables = ['x^2', 'y^2', 'z^2', 'xy', 'yz', 'zx', 'x', 'y', 'z', '']

    for i, coeff in enumerate(coeffs):
        if coeff == 0:
            continue

        var_str = variables[i]

        sign_str = '+' if coeff > 0 else '-'
        coeff_val_abs = abs(coeff)

        coeff_str = ''
        # For LaTeX, coefficient 1 is usually omitted unless it's a constant term.
        if coeff_val_abs != 1 or var_str == '':
            coeff_str = f"{coeff_val_abs}"

        current_term = f"{coeff_str}{var_str}"

        if not terms: # First term, no leading '+'
            if sign_str == '-':
                terms.append(f"-{current_term}")
            else:
                terms.append(current_term)
        else: # Subsequent terms, add sign explicitly
            terms.append(f"{sign_str} {current_term}")

    equation_str = " ".join(terms).replace("+ -", "- ").replace("- -", "+ ").strip()

    if not equation_str:
        return "$0 = 0$"

    # Wrap the entire equation in dollar signs for LaTeX rendering
    return f"${equation_str} = 0$"

def create_quadric_matrix(coeffs):
    """
    Generates the 4x4 symmetric matrix for a quadric surface from its coefficients.
    coeffs: [A, B, C, D, E, F, G, H, I, J]
    """
    A, B, C, D, E, F, G, H, I, J = coeffs

    matrix = np.zeros((4, 4))

    matrix[0, 0] = A
    matrix[1, 1] = B
    matrix[2, 2] = C
    matrix[3, 3] = J

    matrix[0, 1] = matrix[1, 0] = D / 2
    matrix[1, 2] = matrix[2, 1] = E / 2
    matrix[0, 2] = matrix[2, 0] = F / 2

    matrix[0, 3] = matrix[3, 0] = G / 2
    matrix[1, 3] = matrix[3, 1] = H / 2
    matrix[2, 3] = matrix[3, 2] = I / 2

    return matrix

def format_coeff_for_latex(val, is_half=False):
    """
    Formats a coefficient for LaTeX display, handling fractions for '/2' cases.
    """
    if val == 0:
        return '0'
    if is_half:
        if val == 1:
            return '1/2'
        elif val == -1:
            return '-1/2'
        elif val % 2 == 0:
            return str(int(val / 2))
        else:
            return f"{val}/2"
    else:
        return str(val)

def format_matrix_to_latex(coeffs, name='M'):
    """
    Formats the quadric matrix coefficients into a symbolic LaTeX string.
    """
    A, B, C, D, E, F, G, H, I, J = coeffs
    latex_str = rf"${name} = \begin{{pmatrix}}"

    row1_entries = [
        format_coeff_for_latex(A, False),
        format_coeff_for_latex(D, True),
        format_coeff_for_latex(F, True),
        format_coeff_for_latex(G, True)
    ]
    latex_str += " & ".join(row1_entries) + r" \\"

    row2_entries = [
        format_coeff_for_latex(D, True),
        format_coeff_for_latex(B, False),
        format_coeff_for_latex(E, True),
        format_coeff_for_latex(H, True)
    ]
    latex_str += " & ".join(row2_entries) + r" \\"

    row3_entries = [
        format_coeff_for_latex(F, True),
        format_coeff_for_latex(E, True),
        format_coeff_for_latex(C, False),
        format_coeff_for_latex(I, True)
    ]
    latex_str += " & ".join(row3_entries) + r" \\"

    row4_entries = [
        format_coeff_for_latex(G, True),
        format_coeff_for_latex(H, True),
        format_coeff_for_latex(I, True),
        format_coeff_for_latex(J, False)
    ]
    latex_str += " & ".join(row4_entries) + r" \\"

    latex_str += r"\end{pmatrix}$"
    return latex_str

def plot_quadric_surface(coeffs, bounds=(-10, 10), grid_density=60):
    """
    coeffs: [A, B, C, D, E, F, G, H, I, J] 순서의 리스트
    Equation: Ax^2 + By^2 + Cz^2 + Dxy + Eyz + Fzx + Gx + Hy + Iz + J = 0
    Plotly를 사용하여 시각화합니다.
    """
    A, B, C, D, E, F, G, H, I, J = coeffs

    # 1. 3D 그리드 생성
    x, y, z = np.mgrid[bounds[0]:bounds[1]:complex(0, grid_density),
                       bounds[0]:bounds[1]:complex(0, grid_density),
                       bounds[0]:bounds[1]:complex(0, grid_density)]

    # 2. 방정식 계산 (Implicit function value)
    vol = (A * x**2 + B * y**2 + C * z**2 +
           D * x * y + E * y * z + F * z * x +
           G * x + H * y + I * z + J)

    # 3. 등가면(Isosurface) 추출 (Marching Cubes 알고리즘)
    try:
        verts, faces, _, _ = measure.marching_cubes(vol, level=0, spacing=(
            (bounds[1]-bounds[0])/grid_density,
            (bounds[1]-bounds[0])/grid_density,
            (bounds[1]-bounds[0])/grid_density
        ))

        # 좌표 보정 (mgrid 인덱스를 실제 좌표로 변환)
        verts += bounds[0]

    except ValueError:
        print("설정한 범위 내에 그려질 곡면이 없어.")
        return

    # 4. Plotly를 사용한 시각화
    equation_text_latex = get_equation_string(coeffs)
    numerical_quadric_matrix = create_quadric_matrix(coeffs) # Keep for determinant calculation
    matrix_latex = format_matrix_to_latex(coeffs) # Use coeffs for symbolic display

    # Calculate determinants using the numerical matrix
    det_M = np.linalg.det(numerical_quadric_matrix)
    M_tilde = numerical_quadric_matrix[0:3, 0:3] # Extract 3x3 submatrix from quadratic terms
    det_M_tilde = np.linalg.det(M_tilde)

    # Corrected LaTeX strings to suppress SyntaxWarning using raw f-strings
    det_M_latex = rf"$\det(M) = {det_M:.2f}$"
    det_M_tilde_latex = rf"$\det(\tilde{{M}}) = {det_M_tilde:.2f}$"

    # 5. Plotly Figure 생성
    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:, 0],
        y=verts[:, 1],
        z=verts[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color='lightblue',
        opacity=0.50
    )])

    fig.update_layout(
        title=f"Quadric Surface: {equation_text_latex}<br>{matrix_latex}<br>{det_M_latex}, {det_M_tilde_latex}",
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data' # Ensure uniform scaling
        ),
        margin=dict(l=0, r=0, b=0, t=50)
    )

    fig.show()

In [6]:
ellipsoid_coeffs = [1, 1, 1, 0, 0, 0, 0, 0, 0, -25] # x^2 + y^2 + z^2 - 25 = 0 (sphere)
plot_quadric_surface(ellipsoid_coeffs)